# Notebook 0 — Single Extraction Pass (checkpointed)

One pass over the shards produces everything the rest of the study needs:

| Output | Used for |
|---|---|
| `clip_logits` (12 scene classes) | **Task B** labels (teacher) |
| `resnet_emb` (2048-d, ImageNet ResNet50) | the **student** image branch — deliberately a different backbone from the teacher |
| `lat`, `lon` | metadata + **Task A** labels |
| `width`, `height`, `n_bytes` | file-level metadata group |

**Checkpointing.** Kaggle sessions die. This notebook saves one `.npz` per shard and skips
shards already present in any attached checkpoint dataset, so a killed session costs you
one shard, not the whole run. It also stops cleanly before the session limit.

**Resume procedure**
1. Run until it stops (time budget or shards exhausted).
2. *Save Version → Save & Run All*, then publish the output as a Kaggle Dataset.
3. Attach that dataset to this notebook and add its path to `CKPT_INPUTS`.
4. Re-run. It picks up where it left off.

In [ ]:
import os, io, gc, json, time, glob, itertools, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models
from PIL import Image
import msgpack

warnings.filterwarnings('ignore')
Image.MAX_IMAGE_PIXELS = None
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ════════════════════════════════════════════════════════════════════════
SHARD_DIR   = Path('/kaggle/input/large-dataset-of-geotagged-images/shards')
OUT         = Path('/kaggle/working'); OUT.mkdir(exist_ok=True, parents=True)
CKPT_DIR    = OUT / 'shards_done'; CKPT_DIR.mkdir(exist_ok=True, parents=True)

# Add previously-published checkpoint datasets here to resume across sessions.
CKPT_INPUTS = [
    # Path('/kaggle/input/nb0-extract-part1/shards_done'),
]

TARGET_N     = 300_000     # stop once this many images are extracted
RECORD_STRIDE = 1          # >1 subsamples within each shard
BATCH        = 256
NUM_WORKERS  = 2
TIME_BUDGET_H = 8.0        # stop cleanly before Kaggle kills the session

CLIP_LOCAL   = None        # e.g. Path('/kaggle/input/clip-vit-base-patch32')
T0 = time.time()

def elapsed_h():
    return (time.time() - T0) / 3600

## Optional: mirror checkpoints to Google Drive

Kaggle Datasets (already used above) is the durable store — a session dying does not
lose anything already checkpointed. This is a convenience mirror on top: it lets you
browse figures and tables from a normal Drive folder without opening Kaggle.

**One-time setup:** create a Google Cloud service account with the Drive API enabled,
share a Drive folder with its email address as Editor, and add two Kaggle Secrets —
`GDRIVE_SA_JSON` (the service account key, full JSON) and `GDRIVE_FOLDER_ID` (the
folder's ID from its URL). Missing secrets => this cell no-ops safely.

In [ ]:
DRIVE_SYNC = True  # set False to disable without removing the secrets

def get_drive():
    from kaggle_secrets import UserSecretsClient
    from google.oauth2.service_account import Credentials
    from googleapiclient.discovery import build
    us = UserSecretsClient()
    creds = Credentials.from_service_account_info(
        json.loads(us.get_secret('GDRIVE_SA_JSON')),
        scopes=['https://www.googleapis.com/auth/drive.file'])
    return build('drive', 'v3', credentials=creds), us.get_secret('GDRIVE_FOLDER_ID')

def _drive_folder(drive, parent_id, name):
    q = (f"name='{name}' and '{parent_id}' in parents and "
         "mimeType='application/vnd.google-apps.folder' and trashed=false")
    hits = drive.files().list(q=q, fields='files(id)').execute()['files']
    if hits:
        return hits[0]['id']
    return drive.files().create(
        body={'name': name, 'mimeType': 'application/vnd.google-apps.folder',
              'parents': [parent_id]}, fields='id').execute()['id']

def drive_upload(local_path, drive, folder_id, subfolder=None):
    from googleapiclient.http import MediaFileUpload
    parent = _drive_folder(drive, folder_id, subfolder) if subfolder else folder_id
    name = Path(local_path).name
    existing = drive.files().list(
        q=f"name='{name}' and '{parent}' in parents and trashed=false",
        fields='files(id)').execute()['files']
    media = MediaFileUpload(str(local_path), resumable=True)
    if existing:
        drive.files().update(fileId=existing[0]['id'], media_body=media).execute()
    else:
        drive.files().create(body={'name': name, 'parents': [parent]}, media_body=media).execute()

def sync_to_drive(paths, subfolder=None):
    """Best-effort mirror to Drive. Never raises -- a sync failure must not break
    the run. Kaggle Datasets remain the source of truth regardless of this outcome."""
    if not DRIVE_SYNC:
        return
    try:
        drive, folder_id = get_drive()
    except Exception as e:
        print(f'[drive sync skipped: {e}]'); return
    ok = 0
    for p in paths:
        p = Path(p)
        if not p.exists():
            continue
        try:
            drive_upload(p, drive, folder_id, subfolder); ok += 1
        except Exception as e:
            print(f'  [drive] FAILED {p.name}: {e}')
    print(f'  [drive] synced {ok}/{len(paths)} files' + (f' -> {subfolder}/' if subfolder else ''))


In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  SCENE TAXONOMY  (12 classes, all scene-level)
# ════════════════════════════════════════════════════════════════════════
CLASSES = [
    'a beach or coastline', 'a mountain landscape', 'a forest',
    'a desert', 'snow or ice', 'farmland or fields',
    'a lake or river', 'a city street', 'a historic building or monument',
    'an indoor room', 'a park or garden', 'an industrial site',
]
SHORT = ['beach', 'mountain', 'forest', 'desert', 'snow', 'farmland',
         'water', 'street', 'monument', 'indoor', 'park', 'industrial']

TEMPLATES = [
    'a photo of {}.', 'a photograph of {}.', 'a picture of {}.',
    'a snapshot of {}.', 'a photo showing {}.', 'an image of {}.',
]
print(f'{len(CLASSES)} classes x {len(TEMPLATES)} prompt templates')

In [ ]:
# ── Load CLIP teacher (HF hub, else local Kaggle dataset) ───────────────
def load_clip():
    from transformers import CLIPModel, CLIPTokenizer
    src = str(CLIP_LOCAL) if CLIP_LOCAL else 'openai/clip-vit-base-patch32'
    try:
        m = CLIPModel.from_pretrained(src)
        tk = CLIPTokenizer.from_pretrained(src)
        print(f'CLIP loaded from {src}')
        return m, tk
    except Exception as e:
        raise RuntimeError(
            f'Could not load CLIP from {src}: {e}\n'
            'Fix: enable Internet in Settings (needs phone verification), or download '
            'openai/clip-vit-base-patch32 into a Kaggle Dataset and set CLIP_LOCAL.')

clip, tok = load_clip()
clip = clip.to(DEVICE).eval()

# prompt-ensembled text prototypes: average over templates, then renormalise
with torch.no_grad():
    protos = []
    for c in CLASSES:
        t = tok([tpl.format(c) for tpl in TEMPLATES], padding=True, return_tensors='pt').to(DEVICE)
        e = clip.get_text_features(**t)
        e = F.normalize(e, dim=-1).mean(0)
        protos.append(F.normalize(e, dim=-1))
    TEXT = torch.stack(protos)               # (12, D)
LOGIT_SCALE = clip.logit_scale.exp().item()
print('text prototypes:', tuple(TEXT.shape), '| logit scale', round(LOGIT_SCALE, 2))

In [ ]:
# ── Load ResNet50 student backbone (different family from the teacher) ──
rn = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
RN_DIM = rn.fc.in_features
rn.fc = nn.Identity()
rn = rn.to(DEVICE).eval()
for p in rn.parameters():
    p.requires_grad = False
print(f'ResNet50 embedding dim = {RN_DIM}')

CLIP_NORM = T.Normalize((0.48145466, 0.4578275, 0.40821073),
                        (0.26862954, 0.26130258, 0.27577711))
IN_NORM   = T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
BASE = T.Compose([T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
                  T.CenterCrop(224), T.ToTensor()])

In [ ]:
# ── Shard reader + dataset ──────────────────────────────────────────────
def read_shard(path, stride=1):
    """Yield (id, lat, lon, jpeg_bytes) from a msgpack shard."""
    with open(path, 'rb') as f:
        unp = msgpack.Unpacker(f, raw=False, strict_map_key=False)
        for i, rec in enumerate(unp):
            if isinstance(rec, list) and rec and isinstance(rec[0], dict):
                for j, r in enumerate(rec):
                    if j % stride == 0:
                        yield r['id'], r['latitude'], r['longitude'], r['image']
                continue
            if i % stride:
                continue
            yield rec['id'], rec['latitude'], rec['longitude'], rec['image']


class ShardSet(Dataset):
    def __init__(self, records):
        self.rec = records
    def __len__(self):
        return len(self.rec)
    def __getitem__(self, i):
        _id, lat, lon, blob = self.rec[i]
        try:
            img = Image.open(io.BytesIO(blob)).convert('RGB')
            w, h, ok = img.size[0], img.size[1], 1
        except Exception:
            img = Image.new('RGB', (224, 224), (0, 0, 0)); w = h = 0; ok = 0
        x = BASE(img)
        return CLIP_NORM(x), IN_NORM(x), ok, w, h, len(blob), i

In [ ]:
# ── Checkpoint registry ─────────────────────────────────────────────────
def done_shards():
    seen = set()
    for d in [CKPT_DIR] + [Path(p) for p in CKPT_INPUTS]:
        if Path(d).exists():
            seen |= {Path(p).stem for p in glob.glob(str(Path(d) / '*.npz'))}
    return seen

def n_done():
    tot = 0
    for d in [CKPT_DIR] + [Path(p) for p in CKPT_INPUTS]:
        for p in glob.glob(str(Path(d) / '*.npz')):
            with np.load(p) as z:
                tot += len(z['ids'])
    return tot

shards = sorted(glob.glob(str(SHARD_DIR / '*.msg')))
have = done_shards()
print(f'{len(shards)} shards found | {len(have)} already extracted | '
      f'{n_done():,} images banked')

In [ ]:
# ════════════════════════════════════════════════════════════════════════
#  EXTRACT
# ════════════════════════════════════════════════════════════════════════
total = n_done()
for sp in shards:
    stem = Path(sp).stem
    if stem in have:
        continue
    if total >= TARGET_N:
        print(f'target {TARGET_N:,} reached'); break
    if elapsed_h() > TIME_BUDGET_H:
        print(f'time budget hit at {elapsed_h():.2f} h — save a version and resume'); break

    t0 = time.time()
    recs = list(itertools.islice(read_shard(sp, RECORD_STRIDE), TARGET_N - total))
    if not recs:
        continue
    dl = DataLoader(ShardSet(recs), batch_size=BATCH, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)

    n = len(recs)
    L  = np.zeros((n, len(CLASSES)), np.float16)
    E  = np.zeros((n, RN_DIM), np.float16)
    OKk = np.zeros(n, np.int8); W = np.zeros(n, np.int32)
    H = np.zeros(n, np.int32);  B = np.zeros(n, np.int32)

    with torch.no_grad():
        for xc, xi, ok, w, h, nb, idx in dl:
            idx = idx.numpy()
            with torch.autocast('cuda', enabled=DEVICE == 'cuda'):
                f = F.normalize(clip.get_image_features(pixel_values=xc.to(DEVICE)), dim=-1)
                lg = LOGIT_SCALE * f @ TEXT.T
                em = rn(xi.to(DEVICE))
            L[idx] = lg.float().cpu().numpy().astype(np.float16)
            E[idx] = em.float().cpu().numpy().astype(np.float16)
            OKk[idx] = ok.numpy(); W[idx] = w.numpy(); H[idx] = h.numpy(); B[idx] = nb.numpy()

    np.savez_compressed(
        CKPT_DIR / f'{stem}.npz',
        ids=np.array([r[0] for r in recs]),
        lat=np.array([r[1] for r in recs], np.float32),
        lon=np.array([r[2] for r in recs], np.float32),
        clip_logits=L, resnet_emb=E, ok=OKk, width=W, height=H, n_bytes=B)
    sync_to_drive([CKPT_DIR / f'{stem}.npz'], subfolder='nb0_shards_done')

    total += n
    print(f'{stem}: {n:,} imgs in {time.time()-t0:.0f}s | total {total:,}/{TARGET_N:,} '
          f'| {elapsed_h():.2f} h elapsed', flush=True)
    del recs, dl, L, E; gc.collect()

print(f'\nEXTRACTION STATE: {total:,} images across {len(done_shards())} shards')

In [ ]:
# ── Merge checkpoints into the artefacts Notebook A consumes ────────────
if total >= TARGET_N or len(done_shards()) == len(shards):
    parts = []
    for d in [CKPT_DIR] + [Path(p) for p in CKPT_INPUTS]:
        parts += sorted(glob.glob(str(Path(d) / '*.npz')))
    ids, lat, lon, LG, EM, OKa, Wa, Ha, Ba = [], [], [], [], [], [], [], [], []
    for p in parts:
        with np.load(p) as z:
            ids.append(z['ids']); lat.append(z['lat']); lon.append(z['lon'])
            LG.append(z['clip_logits']); EM.append(z['resnet_emb'])
            OKa.append(z['ok']); Wa.append(z['width']); Ha.append(z['height']); Ba.append(z['n_bytes'])

    ids = np.concatenate(ids)
    keep = np.concatenate(OKa).astype(bool)
    _, uniq = np.unique(ids, return_index=True)
    m = np.zeros(len(ids), bool); m[uniq] = True; keep &= m      # drop cross-shard duplicates

    tab = pd.DataFrame({'id': ids[keep],
                        'lat': np.concatenate(lat)[keep],
                        'lon': np.concatenate(lon)[keep],
                        'width': np.concatenate(Wa)[keep],
                        'height': np.concatenate(Ha)[keep],
                        'n_bytes': np.concatenate(Ba)[keep]})
    tab['aspect']   = tab.width / tab.height.clip(lower=1)
    tab['megapix']  = tab.width * tab.height / 1e6
    tab['bytes_px'] = tab.n_bytes / (tab.width * tab.height).clip(lower=1)

    np.save(OUT / 'clip_logits.npy', np.concatenate(LG)[keep])
    np.save(OUT / 'resnet_emb.npy',  np.concatenate(EM)[keep])
    tab.to_parquet(OUT / 'extract_table.parquet', index=False)
    json.dump({'classes': CLASSES, 'short': SHORT, 'templates': TEMPLATES,
               'n': int(len(tab)), 'rn_dim': int(RN_DIM),
               'teacher': 'openai/clip-vit-base-patch32', 'student': 'resnet50-IMAGENET1K_V2',
               'logit_scale': LOGIT_SCALE, 'target_n': TARGET_N},
              open(OUT / 'extract_meta.json', 'w'), indent=2)
    sync_to_drive([OUT / 'clip_logits.npy', OUT / 'resnet_emb.npy',
                   OUT / 'extract_table.parquet', OUT / 'extract_meta.json'],
                  subfolder='nb0_merged_output')
    print(f'MERGED -> {len(tab):,} usable images')
    display(tab.describe().T.round(3))
else:
    print(f'Not complete ({total:,}/{TARGET_N:,}). Save a version, publish '
          f'/kaggle/working/shards_done as a Dataset, add it to CKPT_INPUTS, re-run.')

---
**Next:** publish `clip_logits.npy`, `resnet_emb.npy`, `extract_table.parquet`,
`extract_meta.json` as a Kaggle Dataset and attach to Notebook A.